# Databricks: Author and deploy an MCP tool-calling OpenAI agent

This notebook shows how to author an agentusing the OpenAI SDK that connects to Databricks MCP servers. If you're familiar with OpenAI's API patterns, this provides a direct path to building agents on Databricks infrastructure.

Connect your agent to data and tools through MCP servers. Databricks provides managed MCP servers for Unity Catalog functions, vector search, and Genie spaces. You can also connect to custom MCP servers that you host as Databricks Apps. See [MCP on Databricks](https://docs.databricks.com/aws/en/generative-ai/mcp/).

In this notebook, you:

- Author an [Open AI Responses API](https://platform.openai.com/docs/api-reference/responses) that calls MCP tools
- Connect the agent to MCP servers to access Databricks-hosted tools
- Test the agent and evaluate its responses using MLflow Evaluation
- Log the agent with MLflow and deploy it to a model serving endpoint

This notebook uses the [`ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) interface for compatibility with Databricks AI features like evaluation and deployment.

To learn more about authoring an agent using Agent Framework, see Databricks documentation ([AWS](https://docs.databricks.com/aws/agents/custom-agents/author-agent) | [Azure](https://learn.microsoft.com/azure/databricks/agents/custom-agents/create-chat-model)).

## Prerequisites
- Address all `TODO`s in this notebook.

In [0]:
%pip install --force-reinstall  databricks-openai databricks-agents uv
dbutils.library.restartPython()

## Define the agent in code

Define the agent code in a single cell below. This lets you easily write the agent
code to a local Python file, using the `%%writefile` magic command, for subsequent
logging and deployment.

**What this code does at a high level:**

1. **Connect to MCP servers using adapters**
    The `McpServer` adapter from `databricks_openai.agents.mcp_server` handles:
    - Async connections to Databricks MCP servers
    - Authentication (PAT for managed servers, OAuth/M2M for custom servers)
    - Context management (automatically closes connections when done)

2. **Create an agent with MCP tools**
    The Agents SDK `Agent` class automatically discovers tools from connected
    MCP servers and makes them available for the LLM to call.

3. **Handle streaming responses**
    The `MCPToolCallingAgent` class wraps the Agents SDK agent to:
    - Process streaming events from the OpenAI Responses API
    - Convert events to Databricks-compatible format
    - Enable MLflow tracing for agent execution

4. **Wrap with ResponsesAgent**
    The agent is wrapped using `ResponsesAgent` for compatibility with Databricks
    features like evaluation, deployment, and feedback collection.

5. **MLflow autotracing**
   Enable MLflow autologging to start automatic tracing.

#### Agent tools

This example connects to the Unity Catalog functions MCP server to access
`system.ai.python_exec` (a built-in Python code interpreter). The code also
includes commented-out examples for connecting to:
- Custom MCP servers (hosted as Databricks Apps)
- Vector search MCP servers (with OBO authentication)

In [0]:
%%writefile agent.py

import mlflow
from mlflow.entities import SpanType
from typing import Any, Generator
from databricks_openai import McpServerToolkit, ToolInfo
from databricks.sdk import WorkspaceClient
from mlflow.pyfunc import ResponsesAgent
from databricks_openai import DatabricksOpenAI
import openai
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
import nest_asyncio
import json

nest_asyncio.apply()

############################################
## Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-5"

# TODO: Update with your system prompt
SYSTEM_PROMPT = """
You are a helpful assistant that can run Python code.
"""

# TODO: Choose your MCP server connection type and setup the Workspace Clients for authentication

# ---------------------------------------------------------------------------
# Managed MCP Server — simplest setup
# ---------------------------------------------------------------------------
# Databricks manages this connection automatically using your workspace settings
# and Personal Access Token (PAT) authentication.

workspace_client = WorkspaceClient()

host = workspace_client.config.host

# ---------------------------------------------------------------------------
# Custom MCP Server — hosted as a Databricks App
# ---------------------------------------------------------------------------
# Use this if you’re running your own MCP server in Databricks.
# These require OAuth with a service principal for machine-to-machine (M2M) auth.
#
# Follow the insturctions here in order to create a SP, grant the SP query permissions on your app and then mint a client id and # secret. https://docs.databricks.com/aws/en/dev-tools/auth/oauth-m2m
#
# Uncomment and fill in the settings below to use a custom MCP server.
#
# import os
# custom_mcp_server_workspace_client = WorkspaceClient(
#     host="<DATABRICKS_WORKSPACE_URL>",
#     client_id=os.getenv("DATABRICKS_CLIENT_ID"),
#     client_secret=os.getenv("DATABRICKS_CLIENT_SECRET"),
#     auth_type="oauth-m2m",  # Enables service principal authentication
# )

# ---------------------------------------------------------------------------
# OBO Setup
# ---------------------------------------------------------------------------
# In order to use OBO, uncomment the code below and pass this workspace client to the appropriate McpServer below
#
# from databricks_ai_bridge import ModelServingUserCredentials
# obo_workspace_client = WorkspaceClient(credentials_strategy=ModelServingUserCredentials())

###############################################################################
## Configure MCP Servers for your agent
##
## This section sets up server connections so your agent can retrieve data or take actions.

## There are three connection types:
## 1. Managed MCP servers — fully managed by Databricks
## 2. External MCP servers — hosted outside Databricks but proxied through a
##    Managed MCP server proxy
## 3. Custom MCP servers — MCP servers hosted as Databricks Apps
##
###############################################################################
mcp_servers = [
  McpServerToolkit(url=f"{host}/api/2.0/mcp/functions/system/ai"),
  # McpServerToolkit(url="custom_app_url", workspace_client=custom_mcp_server_workspace_client),
  # McpServerToolkit(url=f"{host}/api/2.0/mcp/ai-search/system/ai", workspace_client=obo_workspace_client)
]

class MCPToolCallingAgent(ResponsesAgent):
  def __init__(self,llm_endpoint: str,mcp_servers: list[McpServerToolkit],):
    self.llm_endpoint = llm_endpoint
    self.workspace_client = WorkspaceClient()
    self.model_serving_client = DatabricksOpenAI()
    self.mcp_servers = mcp_servers
    self.tools_dict = {}

    for mcp_server in mcp_servers:
        tool_infos = mcp_server.get_tools()
        for tool_info in tool_infos:
            if tool_info.name in self.tools_dict:
                raise ValueError(
                    f"Tool Name {tool_info.name} already exists. For MCP Server: {mcp_server.name or mcp_server.url}, specify a new mcp server name to make the tool names unique."
                )
            self.tools_dict[tool_info.name] = tool_info
  
  @mlflow.trace(span_type=SpanType.TOOL)
  def execute_tool(self, tool_name: str, args: dict) -> Any:
      return self.tools_dict[tool_name].execute(**args)
  
  @mlflow.trace(span_type=SpanType.LLM)
  def call_llm(
      self, messages: list[dict[str, Any]]
  ) -> Generator[dict[str, Any], None, None]:
      for chunk in self.model_serving_client.chat.completions.create(
          model=self.llm_endpoint,
          messages=to_chat_completions_input(messages),
          tools=[tool.spec for tool in self.tools_dict.values()],
          stream=True,
      ):
          yield chunk.to_dict()
  
  def handle_tool_call(
    self, tool_call: dict[str, Any], messages: list[dict[str, Any]]
  ) -> ResponsesAgentStreamEvent:
      """
      Execute tool calls, add them to the running message history, and return a ResponsesStreamEvent w/ tool output
      """
      if tool_call["arguments"]:
          args = json.loads(tool_call["arguments"])
      else:
          args = {}
      result = str(self.execute_tool(tool_name=tool_call["name"], args=args))

      tool_call_output = self.create_function_call_output_item(
          tool_call["call_id"], result
      )
      messages.append(tool_call_output)
      return ResponsesAgentStreamEvent(
          type="response.output_item.done", item=tool_call_output
      )

  def call_and_run_tools(
      self,
      messages: list[dict[str, Any]],
      max_iter: int = 10,
  ) -> Generator[ResponsesAgentStreamEvent, None, None]:
      for _ in range(max_iter):
          last_msg = messages[-1]
          if last_msg.get("role", None) == "assistant":
              return
          elif last_msg.get("type", None) == "function_call":
              yield self.handle_tool_call(last_msg, messages)
          else:
              yield from output_to_responses_items_stream(
                  chunks=self.call_llm(messages), aggregator=messages
              )

      yield ResponsesAgentStreamEvent(
          type="response.output_item.done",
          item=self.create_text_output_item(
              "Max iterations reached. Stopping.", str(uuid4())
          ),
      )

  def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
      outputs = [
          event.item
          for event in self.predict_stream(request)
          if event.type == "response.output_item.done"
      ]
      return ResponsesAgentResponse(
          output=outputs, custom_outputs=request.custom_inputs
      )

  def predict_stream(
      self, request: ResponsesAgentRequest
  ) -> Generator[ResponsesAgentStreamEvent, None, None]:
      messages = [i.model_dump() for i in request.input]

      yield from self.call_and_run_tools(messages)

# Log the model using MLflow
mlflow.openai.autolog()
AGENT = MCPToolCallingAgent(llm_endpoint=LLM_ENDPOINT_NAME, mcp_servers=mcp_servers)
mlflow.models.set_model(AGENT)

## Test the agent

Interact with the agent to test its output. Since we manually traced methods within `ResponsesAgent`, you can view the trace for each step the agent takes, with any LLM calls made via the OpenAI SDK automatically traced by autologging.

Replace this placeholder input with an appropriate domain-specific example for your agent.

In [0]:
dbutils.library.restartPython()

In [0]:
# ==============================================================================
# TODO: ONLY UNCOMMENT AND EDIT THIS SECTION IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
#       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
# ==============================================================================

# # Set your Databricks client ID and client secret for service principal authentication.
# DATABRICKS_CLIENT_ID = "<YOUR_CLIENT_ID>"
# client_secret_scope_name = "<YOUR_SECRET_SCOPE>"
# client_secret_key_name = "<YOUR_SECRET_KEY_NAME>"

# # Load your service principal credentials into environment variables
# os.environ["DATABRICKS_CLIENT_ID"] = DATABRICKS_CLIENT_ID
# os.environ["DATABRICKS_CLIENT_SECRET"] = dbutils.secrets.get(scope=client_secret_scope_name, key=client_secret_key_name)

In [0]:
from agent import AGENT

result = AGENT.predict({"input": [{"role": "user", "content": "What is 6*7 in Python"}]})
print(result.model_dump(exclude_none=True))

In [0]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "What is 6*7 in Python?"}]}
):
    print(chunk.model_dump(exclude_none=True))

## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

### Enable automatic authentication for Databricks resources
For the most common Databricks resource types, Databricks supports and recommends declaring resource dependencies for the agent upfront during logging. This enables automatic authentication passthrough when you deploy the agent. With automatic authentication passthrough, Databricks automatically provisions, rotates, and manages short-lived credentials to securely access these resource dependencies from within the agent endpoint.

To enable automatic authentication, specify the dependent Databricks resources when calling `mlflow.pyfunc.log_model().`

  - **TODO**: If your Unity Catalog tool queries a [vector search index](docs link) or leverages [external functions](docs link), you need to include the dependent vector search index and UC connection objects, respectively, as resources. See docs ([AWS](https://docs.databricks.com/agents/custom-agents/model-serving/agent-authentication-model-serving.html#log-resources-automatic) | [Azure](https://learn.microsoft.com/azure/databricks/agents/custom-agents/model-serving/log-agent#resources)).

In [0]:
import mlflow
from agent import LLM_ENDPOINT_NAME
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
from pkg_resources import get_distribution

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME), DatabricksFunction(function_name="system.ai.python_exec")]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            "backoff",
            f"mlflow=={get_distribution('mlflow').version}",
            f"mcp=={get_distribution('mcp').version}",
            f"databricks-openai=={get_distribution('databricks-openai').version}"
        ]
    )

## Evaluate the agent with Agent Evaluation

Use Agent Evaluation to evalaute the agent's responses based on expected responses and other evaluation criteria. Use the evaluation criteria you specify to guide iterations, using MLflow to track the computed quality metrics.
See Databricks documentation ([AWS]((https://docs.databricks.com/aws/generative-ai/agent-evaluation) | [Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-evaluation/)).


To evaluate your tool calls, add custom metrics. See Databricks documentation ([AWS](https://docs.databricks.com/en/generative-ai/agent-evaluation/custom-metrics.html#evaluating-tool-calls) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-evaluation/custom-metrics#evaluating-tool-calls)).

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness, RetrievalRelevance, Safety

eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Calculate the 15th Fibonacci number"}]},
        "expected_response": "The 15th Fibonacci number is 610.",
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],  # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}]},
    env_manager="uv",
)

## Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

- **TODO** Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "peterlu"
schema = "default"
model_name = "mcp-openai-test"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

## Deploy the agent

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME, 
    uc_registered_model_info.version,
    environment_vars={
    # ==============================================================================
    # TODO: ONLY UNCOMMENT AND CONFIGURE THE ENVIRONMENT_VARS SECTION BELOW
    #       IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
    #       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
    # ==============================================================================
    #     "DATABRICKS_CLIENT_ID": DATABRICKS_CLIENT_ID,
    #     "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{client_secret_scope_name}/{client_secret_key_name}}}}}"
    },
    tags = {"endpointSource": "docs"},
    deploy_feedback_model=False
)

## Next steps

After your agent is deployed, you can chat with it in AI playground to perform additional checks, share it with SMEs in your organization for feedback, or embed it in a production application. See docs ([AWS](https://docs.databricks.com/en/generative-ai/deploy-agent.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/deploy-agent)) for details